# Manuscript tables: programmatic generation and validation
-------

Sources: `results/performance_matrix/`, `results/main_nested_cv/`, `results/comparison/`,
`results/mapping/`, `results/autocorrelation/`, `results/spatial_baseline/`,
`results/spatial_buffer/`, `results/coordinate_cnn_buffer/`,
`data/cache/performance_matrix_buffered/`, and `figures/014_robustness_checks/tables/`.

In [ ]:
NOTEBOOK = "015_manuscript_tables"

import json
import re

import geopandas as gpd
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from utils.paths import get_project_paths
from utils.terminology import (
    BOOTSTRAP_REPS_CROSS_CONFIG,
    BOOTSTRAP_REPS_INFERENCE,
    COMPARISON_STUDIES,
    FEATURE_SET_BANDS,
    FEATURE_SETS,
    FOLD_IDS,
    NODATA,
    SEED,
    SPECIES_GROUP,
)

paths = get_project_paths()
LATEX_DIR = paths.figures / NOTEBOOK / "latex"
LATEX_DIR.mkdir(parents=True, exist_ok=True)
for _stale in LATEX_DIR.glob("*.tex"):  # the run regenerates the full snippet set
    _stale.unlink()

# Canonical runs (newest of each role).
PM_RUN = sorted(p for p in (paths.results / "performance_matrix").iterdir() if p.is_dir())[-1]
COMP_RUN = sorted(p for p in (paths.results / "comparison").iterdir() if p.is_dir())[-1]
MAPPING_RUN = sorted(p for p in (paths.results / "mapping").iterdir() if p.is_dir())[-1]
NESTED = paths.results / "main_nested_cv"
XY = "xy_coords"
XY_LABEL = "Coordinate-only (x, y)"
FS_ORDER = ["baseline", "baseline_conventional_eo", "baseline_alphaearth", "baseline_tessera"]
FS_SHORT = {
    "baseline": "Baseline",
    "baseline_conventional_eo": "+ Conventional EO",
    "baseline_alphaearth": "+ AlphaEarth",
    "baseline_tessera": "+ TESSERA",
    XY: XY_LABEL,
}
ARCH_ORDER = ["xgboost", "cnn_3x3", "cnn_5x5", "cnn_7x7"]
ARCH_TEX = {
    "xgboost": "XGBoost",
    "cnn_3x3": r"CNN 3$\times$3",
    "cnn_5x5": r"CNN 5$\times$5",
    "cnn_7x7": r"CNN 7$\times$7",
}
BUFFERS = (0, 10, 20)

# ------------------------------------------------------------------ validation ledger
CHECKS = []


def check(name, passed, detail=""):
    """Record one deterministic internal-consistency check and print its verdict."""
    CHECKS.append({"check": name, "passed": bool(passed), "detail": detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f" ({detail})" if detail else ""))


def max_abs_diff(pairs):
    """max |a - b| over an iterable of (a, b) pairs, as a printable detail string."""
    worst = max((abs(float(a) - float(b)) for a, b in pairs), default=0.0)
    return worst, f"max |diff| = {worst:.2e}"


# ------------------------------------------------------------------ formatting helpers
def f2(v):
    return "--" if pd.isna(v) else f"{v:.2f}"


def f3(v):
    return "--" if pd.isna(v) else f"{v:.3f}"


def ha0(v):
    """Hectares with LaTeX thousands separators: 53896.1 -> '53{,}896'."""
    return f"{v:,.0f}".replace(",", "{,}")


def ci3(v, lo, hi):
    return f"{f3(v)} [{f3(lo)}, {f3(hi)}]"


def pfmt(p):
    if pd.isna(p):
        return "--"
    return f"{p:.4f}" if p < 0.01 else f"{p:.3f}"


def texnum(cell):
    """LaTeX minus signs in a numeric cell string."""
    return cell.replace("-", "$-$")


def tex_caption(label):
    """A caption holding only the table's label, e.g. ``tab:sup_hp_gaps``.

    The descriptive captions and notes live in the manuscript, not in the snippets.
    """
    return "\\caption{" + label.replace("_", "\\_") + "}"


def write_tex(label, title, latex, show=True):
    """Save one table's LaTeX under its label, headed by an identifying comment banner."""
    rule = "% " + "=" * 76
    banner = f"{rule}\n% {title}\n% label: {label}\n{rule}\n"
    path = LATEX_DIR / f"{label.split(':')[-1]}.tex"
    path.write_text(banner + latex, encoding="utf-8")
    if show:
        print(f"\n--- copy-and-paste LaTeX for {label} (saved to {path.name}) ---")
        print(banner + latex)
    else:
        print(f"[latex] {title} -> {path}")
    return path


# --------------------------------------------------------------------- shared loaders
def headline_run(arch, fs):
    """Newest complete main nested-CV run (all folds' parcel predictions present)."""
    root = paths.results / "spatial_baseline" if (fs == XY and arch == "xgboost") else NESTED
    for run in sorted(root.glob(f"*__{arch}__{fs}"), reverse=True):
        if all((run / f"parcel_predictions_fold{f}.parquet").is_file() for f in FOLD_IDS):
            return run
    raise FileNotFoundError(f"no complete run for {arch}/{fs}")


def parcel_frames(arch, fs, buffer_km):
    """{fold: out-of-fold parcel predictions} for one configuration and buffer."""
    run = headline_run(arch, fs)
    if fs == XY and arch == "xgboost" and buffer_km > 0:
        runs = [
            r
            for r in sorted((paths.results / "spatial_buffer").glob(f"*__xgboost__{XY}"))
            if (r / "run_arms.json").is_file()
            and json.loads((r / "run_arms.json").read_text(encoding="utf-8"))["arms"]
            == ["buffered"]
        ]
        frames = {}
        for f in FOLD_IDS:
            ck = pd.read_parquet(runs[-1] / "fold_checkpoints" / f"fold{f}_parcel.parquet")
            sub = ck[(ck["arm"] == "buffered") & (ck["gap_km"] == buffer_km)]
            frames[f] = sub[["parcel_id", "outer_fold", "y_true", "p_mean", "n_pixels"]]
        return frames
    if buffer_km == 0:
        return {f: pd.read_parquet(run / f"parcel_predictions_fold{f}.parquet") for f in FOLD_IDS}
    cdir = paths.cache / "performance_matrix_buffered" / f"{arch}__{fs}__{run.name}__{buffer_km}km"
    return {f: pd.read_parquet(cdir / f"buffered_fold{f}.parquet") for f in FOLD_IDS}


def pooled_frame(frames):
    return pd.concat(
        [fr.assign(outer_fold=int(f)) for f, fr in sorted(frames.items())], ignore_index=True
    )


def inner_thresholds(arch, fs):
    """Per-fold parcel inner thresholds (best HP trial, or per_fold_metrics fallback)."""
    run = headline_run(arch, fs)
    if all((run / f"hp_trials_fold{f}.csv").is_file() for f in FOLD_IDS):
        thresholds = {}
        for f in FOLD_IDS:
            trials = pd.read_csv(run / f"hp_trials_fold{f}.csv")
            complete = trials[trials["state"] == "COMPLETE"]
            thresholds[f] = float(
                complete.loc[complete["pr_auc"].idxmax(), "inner_parcel_threshold"]
            )
        return thresholds
    pfm = pd.read_csv(run / "per_fold_metrics.csv")
    pfm = pfm[pfm["level"] == "parcel"]
    return {int(r["outer_fold"]): float(r["inner_threshold"]) for _, r in pfm.iterrows()}


_labels_gdf = gpd.read_file(paths.labels / "ogf_reference_labels_partitioned.gpkg")
_labels_gdf = _labels_gdf[_labels_gdf["ogf"].notna()]
BLOCK_ASSIGNMENT = pd.DataFrame(
    {
        "parcel_id": _labels_gdf["parcel_id"].astype(np.int64),
        "bootstrap_id": _labels_gdf["bootstrap_id"].astype(np.int64),
    }
)

matrix = pd.read_csv(PM_RUN / "performance_matrix.csv")
mat = matrix.set_index(["feature_set", "architecture", "buffer_km"])
print(
    f"[setup] performance matrix {PM_RUN.name}, comparison {COMP_RUN.name}, "
    f"mapping {MAPPING_RUN.name}; LaTeX snippets -> {LATEX_DIR}"
)

In [ ]:
# Main Table 3: existing-study areas and reference-label agreement, plus this study.
# Internal validation: the saved overall_performance metrics are recomputed from the
# common evaluation parcels, and the matrix rows from the saved parcel predictions.
overall = pd.read_csv(COMP_RUN / "overall_performance.csv").set_index("product")
comp_gpkg = gpd.read_file(COMP_RUN / "product_parcel_comparison.gpkg")
comp_gpkg["area_ha"] = comp_gpkg.geometry.area / 1e4
STUDY_ORDER = ["sabatini", "munteanu", "kathmann", "schickhofer"]
parcel_area = {
    s: float(comp_gpkg.loc[comp_gpkg[f"ogf_{s}"].astype(bool), "area_ha"].sum())
    for s in STUDY_ORDER
}
aoi_ha = float(gpd.read_file(paths.aoi).geometry.area.sum() / 1e4)
# The predicted-area shares are quoted against the forested AOI (CORINE Land Cover
# forest classes 311/312/313 clipped to the AOI, as in notebook 006), not the whole
# AOI, because every product only ever maps forest.
forest_ha = float(
    gpd.read_file(
        paths.processed / "vectors" / "corine_land_cover" / "corine_forest_aoi_3035.gpkg"
    ).geometry.area.sum()
    / 1e4
)
mapping_meta = json.loads((MAPPING_RUN / "run_metadata.json").read_text(encoding="utf-8"))
study_area_ha = float(mapping_meta["predicted_ogf_area_parcel_ha"])

# Check 1: threshold-free product metrics recomputed from the evaluation parquet.
ev_check = pd.read_parquet(COMP_RUN / "product_eval_parcels.parquet")
# Extent of the labelled sample itself, for the caption: the parcel area the reference labels
# cover, on the same geometric basis as the AOI and CORINE forest areas quoted beside it.
labelled_ha = float(
    comp_gpkg.loc[
        comp_gpkg["parcel_id"].astype("int64").isin(ev_check["parcel_id"].astype("int64")),
        "area_ha",
    ].sum()
)
y_ev = ev_check["reference_label"].astype(int).to_numpy()
pairs = []
for s in STUDY_ORDER:
    frac = ev_check[f"ogf_{s}_frac"].fillna(0.0).to_numpy()
    pairs.append((average_precision_score(y_ev, frac), overall.loc[s, "pr_auc"]))
    pairs.append((roc_auc_score(y_ev, frac), overall.loc[s, "roc_auc"]))
worst, detail = max_abs_diff(pairs)
check("Main Table 3: product PR/ROC-AUC recomputed from evaluation parcels", worst < 1e-9, detail)

# Check 2: this study's matrix rows recomputed from the saved parcel predictions.
tess = matrix[
    (matrix["feature_set"] == "baseline_tessera") & (matrix["architecture"] == "xgboost")
].set_index("buffer_km")
pairs = []
for buffer_km in (0, 10):
    pooled = pooled_frame(parcel_frames("xgboost", "baseline_tessera", buffer_km))
    y, p = pooled["y_true"].to_numpy(), pooled["p_mean"].to_numpy()
    pairs.append((average_precision_score(y, p), tess.loc[buffer_km, "pr_auc__pooled"]))
    pairs.append((roc_auc_score(y, p), tess.loc[buffer_km, "roc_auc__pooled"]))
worst, detail = max_abs_diff(pairs)
check(
    "Main Table 3: this study's pooled PR/ROC-AUC recomputed from predictions", worst < 1e-9, detail
)

# Informational cross-source comparison (definitions differ slightly by construction).
gpkg_area = float(comp_gpkg.loc[comp_gpkg["ogf_ratsakatika"].astype(bool), "area_ha"].sum())
print(
    f"[info] this study's OGF area: mapping run {study_area_ha:,.0f} ha vs comparison "
    f"gpkg geometry {gpkg_area:,.0f} ha "
    f"({100 * (gpkg_area - study_area_ha) / study_area_ha:+.2f}%)."
)

METRICS_T3 = ["pr_auc", "roc_auc", "f1_inner", "precision_inner", "recall_inner"]
STUDY_META = {
    "sabatini": (
        r"\textcite{sabatini_spatially-explicit_2020}",
        "Europe",
        "Model-based",
        "Primary",
        "0.65",
    ),
    "munteanu": (
        r"\textcite{munteanu_using_2022}",
        "Romania",
        "Model-based",
        "High-conservation-value",
        "0.86",
    ),
    "kathmann": (
        r"\textcite{kathmann_potential_2017}",
        "Romania",
        "Rule-based",
        r"Primary \& old-growth",
        "--",
    ),
    "schickhofer": (
        r"\textcite{schickhofer_inventory_2019}",
        "Romania",
        "Rule-based",
        r"Primary \& old-growth",
        "--",
    ),
}

rows = []
for s in STUDY_ORDER:
    cite, extent, method, target, published = STUDY_META[s]
    o = overall.loc[s]
    cells = [
        ha0(parcel_area[s]),
        f"{100 * parcel_area[s] / forest_ha:.1f}",
        f2(o["pr_auc"]),
        f2(o["roc_auc"]),
        f2(o["f1"]),
        f2(o["precision"]),
        f2(o["recall"]),
        published,
    ]
    rows.append("            " + " & ".join([cite, extent, method, target, *cells]) + r" \\")

study_pct_forest = 100 * study_area_ha / forest_ha
m0, m10 = tess.loc[0], tess.loc[10]
cells0 = [f2(m0[f"{m}__pooled"]) for m in METRICS_T3]
cells10 = [f2(m10[f"{m}__pooled"]) for m in METRICS_T3]
rows.append(
    r"            This study (0\,km buffer)  & \multirow{2}{*}{Făgăraș} & "
    r"\multirow{2}{*}{Model-based} & \multirow{2}{*}{Old-growth} & "
    rf"{ha0(study_area_ha)} & {study_pct_forest:.1f} & " + " & ".join(cells0) + r" & -- \\"
)
rows.append(
    # The predicted area belongs to the deployed 0 km model; the buffered arm is an
    # evaluation configuration and maps nothing, so its two area cells are dashed rather
    # than merged with the row above.
    r"            This study (10\,km buffer) &  &  &  & -- & -- & "
    + " & ".join(cells10)
    + r" & -- \\"
)

t3_latex = (
    r"""\begin{table*}
    \centering
    \caption{tab:existing\_studies\_aoi\_area\_performance}
    \label{tab:existing_studies_aoi_area_performance}
    \setlength{\tabcolsep}{2.5pt}
    \resizebox{\textwidth}{!}{%
    \begin{threeparttable}
        \begin{tabular}{llllcccccccc}
            \toprule
            \multirow{2}{*}[-2.5pt]{Study} & \multirow{2}{*}[-2.5pt]{Extent} & \multirow{2}{*}[-2.5pt]{Method} & \multirow{2}{*}[-2.5pt]{Forest target} & \multicolumn{2}{c}{Predicted area} & \multicolumn{5}{c}{Agreement with Făgăraș reference labels} & \multirow{2}{*}[-2.5pt]{\shortstack{Published\\ROC-AUC}} \\
            \cmidrule(lr){5-6} \cmidrule(lr){7-11}
            & & & & ha & \% forested AOI & PR-AUC & ROC-AUC & F1 & Precision & Recall & \\
            \midrule
"""
    + "\n".join(rows[:4])
    + "\n            \\midrule\n"
    + "\n".join(rows[4:])
    + r"""
            \bottomrule
        \end{tabular}
    \end{threeparttable}%
    }
\end{table*}
"""
)
write_tex(
    "tab:existing_studies_aoi_area_performance",
    "Main Table 3 - Existing-study extent and reference-label agreement",
    t3_latex,
)

t3_show = pd.DataFrame(
    {
        "study": [*STUDY_ORDER, "this study (0 km)", "this study (10 km)"],
        "area_ha": [*(round(parcel_area[s]) for s in STUDY_ORDER), round(study_area_ha), None],
        "pct_forest_aoi": [
            *(round(100 * parcel_area[s] / forest_ha, 1) for s in STUDY_ORDER),
            round(study_pct_forest, 1),
            None,
        ],
        "pr_auc": [
            *(round(overall.loc[s, "pr_auc"], 2) for s in STUDY_ORDER),
            round(m0["pr_auc__pooled"], 2),
            round(m10["pr_auc__pooled"], 2),
        ],
        "roc_auc": [
            *(round(overall.loc[s, "roc_auc"], 2) for s in STUDY_ORDER),
            round(m0["roc_auc__pooled"], 2),
            round(m10["roc_auc__pooled"], 2),
        ],
        "f1": [
            *(round(overall.loc[s, "f1"], 2) for s in STUDY_ORDER),
            round(m0["f1_inner__pooled"], 2),
            round(m10["f1_inner__pooled"], 2),
        ],
    }
)
print("\n[Main Table 3 recomputed]")
print(t3_show.to_string(index=False))

In [ ]:
# Spatial-dependence longtables: every variable entering the models (with per-group
# summary rows) and the out-of-fold residuals (including the coordinate-only control,
# computed here with the same utils.autocorrelation primitives as notebook 007).
from utils.autocorrelation import fit_variograms, moran_correlogram

vf = pd.read_csv(paths.results / "autocorrelation" / "variogram_fits.csv")
mc = pd.read_csv(paths.results / "autocorrelation" / "moran_correlogram.csv")
MODEL_ABBR = {"spherical": "Sph.", "exponential": "Exp.", "gaussian": "Gau."}
MODEL_ORDER = ["spherical", "exponential", "gaussian"]
MORAN_BANDS = [5000, 10000, 15000, 20000]


def _resolved(row):
    eff = row["effective_range_m"]
    maxlag, lag_width = row["maxlag_m"], row["maxlag_m"] / row["n_lags"]
    return not (row["range_hits_maxlag"] or pd.isna(eff) or eff >= 0.9 * maxlag or eff < lag_width)


def variable_stats(fits_var, moran_var):
    """Per-variable numbers: resolved ranges per model, best model, ratio, Moran values."""
    by_model = {r["model"]: r for _, r in fits_var.iterrows()}
    ranges = {
        m: (by_model[m]["effective_range_m"] / 1000 if _resolved(by_model[m]) else np.nan)
        for m in MODEL_ORDER
    }
    resolved = [m for m in MODEL_ORDER if not pd.isna(ranges[m])]
    best = min(resolved, key=lambda m: by_model[m]["rmse"]) if resolved else None
    ratio = by_model[best]["nugget_sill_ratio"] if best else np.nan
    morans = {}
    for band in MORAN_BANDS:
        r = moran_var[moran_var["threshold_m"] == band].iloc[0]
        morans[band] = (float(r["morans_i"]), float(r["p_sim"]))
    return {
        "n": int(fits_var["n"].iloc[0]),
        "ranges": ranges,
        "best": best,
        "ratio": ratio,
        "morans": morans,
    }


def spatial_row(name_tex, stats):
    cells = [name_tex, str(stats["n"])]
    cells += [f2(stats["ranges"][m]) for m in MODEL_ORDER]
    cells.append(MODEL_ABBR[stats["best"]] if stats["best"] else "--")
    cells.append(f2(stats["ratio"]))
    for band in MORAN_BANDS:
        value, p_sim = stats["morans"][band]
        cells.append(f"{value:.3f}" + (r"$^\dagger$" if p_sim >= 0.05 else ""))
    return " & ".join(cells) + r" \\"


SUMMARY_STATS = [
    ("Median", lambda a: np.median(a)),
    ("Min", np.min),
    ("Max", np.max),
]


def summary_rows(stats_list, subtitle):
    """Median/Q1/Q3/IQR/Min/Max rows over a set of variables' resolved ranges, best-model
    nugget:sill ratios and Moran's I values."""
    columns = {
        m: [s["ranges"][m] for s in stats_list if not pd.isna(s["ranges"][m])] for m in MODEL_ORDER
    }
    ratios = [s["ratio"] for s in stats_list if not pd.isna(s["ratio"])]
    morans = {band: [s["morans"][band][0] for s in stats_list] for band in MORAN_BANDS}
    rows = [f"\\multicolumn{{11}}{{l}}{{\\textit{{{subtitle}}}}}\\\\"]
    for stat_name, fn in SUMMARY_STATS:
        cells = [f"\\quad\\textit{{{stat_name}}}", ""]
        cells += [f2(fn(columns[m])) if columns[m] else "--" for m in MODEL_ORDER]
        cells.append("")
        cells.append(f2(fn(ratios)) if ratios else "--")
        cells += [f"{fn(morans[band]):.3f}" for band in MORAN_BANDS]
        rows.append(" & ".join(cells) + r" \\")
    return rows


def longtable_latex(label, sections):
    head = (
        "\\begingroup\n\\footnotesize\n\\setlength{\\tabcolsep}{3pt}\n"
        "\\begin{longtable}{lrrrrlrrrrr}\n" + tex_caption(label) + f"\\label{{{label}}}\\\\\n"
        "\\toprule\n"
        " & & \\multicolumn{3}{c}{Effective range (km)} & & & "
        "\\multicolumn{4}{c}{Moran's $I$ at distance band} \\\\\n"
        "\\cmidrule(lr){3-5}\\cmidrule(lr){8-11}\n"
        "Variable & $n$ & Sph. & Exp. & Gau. & Best & Nug.:sill & 5 km & 10 km & 15 km & "
        "20 km \\\\\n\\midrule\n\\endfirsthead\n"
        f"\\multicolumn{{11}}{{l}}{{\\footnotesize\\textit{{Table \\ref{{{label}}} "
        "continued from the previous page.}}\\\\\n\\toprule\n"
        " & & \\multicolumn{3}{c}{Effective range (km)} & & & "
        "\\multicolumn{4}{c}{Moran's $I$ at distance band} \\\\\n"
        "\\cmidrule(lr){3-5}\\cmidrule(lr){8-11}\n"
        "Variable & $n$ & Sph. & Exp. & Gau. & Best & Nug.:sill & 5 km & 10 km & 15 km & "
        "20 km \\\\\n\\midrule\n\\endhead\n\\midrule\n"
        "\\multicolumn{11}{r}{\\footnotesize\\textit{Continued on the next page.}}\\\\\n"
        "\\endfoot\n\\bottomrule\n\\endlastfoot\n"
    )
    parts = []
    for title, k, rows in sections:
        parts.append(f"\\multicolumn{{11}}{{l}}{{\\textbf{{{title}}} ($k = {k}$)}}\\\\[1pt]")
        parts += rows
        parts.append("\\midrule")
    body = "\n".join(parts[:-1])  # no rule after the last section
    return head + body + "\n\\end{longtable}\n\\endgroup\n"


def var_stats_for(variable):
    return variable_stats(vf[vf["variable"] == variable], mc[mc["variable"] == variable])


# ---- Variables table: label and predictors, with per-group summary rows. --------------
GROUPS = [
    ("Old-growth reference label", "label", False, None, None),
    ("Baseline predictors", "baseline", False, None, "Group summary"),
    ("Conventional Earth observation predictors", "conventional_eo", False, None, "Group summary"),
    (
        "AlphaEarth embeddings (principal components only)",
        "alphaearth",
        True,
        lambda v: "\\texttt{" + v.split("_")[-1] + "}",
        "Summary over all embedding dimensions",
    ),
    (
        "TESSERA embeddings  (principal components only)",
        "tessera",
        True,
        lambda v: "\\texttt{" + v.split("_")[-1] + "}",
        "Summary over all embedding dimensions",
    ),
]
sections = []
for title, group, pc, display, subtitle in GROUPS:
    sub = vf[vf["group"] == group]
    shown_vars = sorted((sub[sub["is_pc"]] if pc else sub[~sub["is_pc"]])["variable"].unique())
    rows = []
    for v in shown_vars:
        name = display(v) if display else "\\texttt{" + v.replace("_", "\\_") + "}"
        rows.append(spatial_row(name, var_stats_for(v)))
    if subtitle:
        summary_vars = sorted(sub[~sub["is_pc"]]["variable"].unique())
        rows += summary_rows(
            [var_stats_for(v) for v in summary_vars], f"{subtitle} (resolved ranges only)"
        )
    sections.append((title, len(shown_vars), rows))

conv_expected = set(FEATURE_SET_BANDS["baseline_conventional_eo"]) - set(
    FEATURE_SET_BANDS["baseline"]
)
check(
    "Spatial-dependence table: analysed variables cover the model feature sets",
    set(vf.loc[(vf["group"] == "baseline") & (~vf["is_pc"]), "variable"])
    == set(FEATURE_SET_BANDS["baseline"])
    and set(vf.loc[(vf["group"] == "conventional_eo") & (~vf["is_pc"]), "variable"])
    == conv_expected,
    f"baseline k={sections[1][1]}, conventional EO k={sections[2][1]}",
)

s3_latex = longtable_latex(
    "tab:sup_autocorrelation_full",
    sections,
)
write_tex(
    "tab:sup_autocorrelation_full",
    "Supplementary table - Spatial dependence of labels and predictors",
    s3_latex,
    show=False,
)
print(
    "[note] longtable environments cannot be wrapped in \\resizebox (they break across "
    "pages); the two spatial-dependence tables keep their \\footnotesize sizing."
)

# ---- Residual longtable, including the coordinate-only control. -----------------------
RES_FS = {
    "baseline": "Baseline",
    "baseline_alphaearth": "AlphaEarth",
    "baseline_conventional_eo": "Conv.\\ EO",
    "baseline_tessera": "TESSERA",
    XY: "Coordinate-only (x, y)",
}

pixels_xy = pd.read_parquet(
    NESTED / "pixel_index.parquet", columns=["pixel_id", "parcel_id", "x", "y"]
)
centroids = pixels_xy.groupby("parcel_id")[["x", "y"]].mean()


def xy_residual_stats(arch):
    """Residual variogram and Moran statistics for one coordinate-only run."""
    pooled = pooled_frame(parcel_frames(arch, XY, 0)).drop_duplicates("parcel_id")
    joined = pooled.join(centroids, on="parcel_id").dropna(subset=["x", "y"])
    coords = joined[["x", "y"]].to_numpy(dtype=float)
    values = (joined["y_true"] - joined["p_mean"]).to_numpy(dtype=float)
    result = fit_variograms(coords, values)
    fits = result.fits.copy()
    fits["n"], fits["maxlag_m"], fits["n_lags"] = result.n, result.maxlag_m, result.n_lags
    moran = moran_correlogram(coords, values).reset_index()
    return variable_stats(fits, moran)


expected_vars = {f"{a}__{fs}__parcel": (a, fs) for a in ARCH_ORDER for fs in FS_ORDER}
present = set(vf.loc[vf["group"] == "residual", "variable"])
missing = sorted(set(expected_vars) - present)
check(
    "Residual table: results/autocorrelation covers all architecture x feature-set runs",
    not missing,
    f"{len(expected_vars) - len(missing)}/{len(expected_vars)} present"
    + (f"; missing {missing}" if missing else ""),
)
res_stats = {
    key: variable_stats(
        vf[(vf["group"] == "residual") & (vf["variable"] == v)],
        mc[(mc["group"] == "residual") & (mc["variable"] == v)],
    )
    for v, key in expected_vars.items()
    if v in present
}
print("[residuals] computing coordinate-only residual autocorrelation (4 architectures)...")
for arch in ARCH_ORDER:
    res_stats[(arch, XY)] = xy_residual_stats(arch)

res_keys = sorted(res_stats, key=lambda k: f"{k[0]}__{k[1]}")
res_rows = [spatial_row(f"{ARCH_TEX[a]}, {RES_FS[f]}", res_stats[(a, f)]) for a, f in res_keys]
res_latex = longtable_latex(
    "tab:sup_autocorrelation_residuals",
    [("Out-of-fold model residuals", len(res_keys), res_rows)],
)
write_tex(
    "tab:sup_autocorrelation_residuals",
    "Supplementary table - Residual spatial dependence",
    res_latex,
    show=False,
)
print(f"[residual table] {len(res_keys)} configurations.")

In [ ]:
# Supplementary table: the numbers behind the block-count sensitivity figure
# (ci_vs_blocks.pdf). Only the bootstrap blocking changes, so the point estimate is fixed
# and the table shows how the interval responds to the number of spatial blocks.
from utils.terminology import BOOTSTRAP_N_UNITS

BLOCK_RUN = paths.results / "block_size_sensitivity"
blocks = pd.read_csv(BLOCK_RUN / "block_size_sensitivity.csv")
block_meta = json.loads((BLOCK_RUN / "run_metadata.json").read_text(encoding="utf-8"))
BLOCK_SOURCE = {
    "folds": "Cross-validation folds",
    "spatial_partition": "Spatial partition",
    "saved_partition": "Saved partition",
}

check(
    "Block-size table: each row realises the requested number of distinct blocks",
    bool((blocks["n_blocks"] == blocks["n_distinct_blocks"]).all()),
    f"counts {list(blocks['n_blocks'])}",
)
check(
    "Block-size table: the headline block count is included and uses the saved partition",
    bool(
        (
            blocks.loc[blocks["n_blocks"] == BOOTSTRAP_N_UNITS, "block_source"] == "saved_partition"
        ).all()
        and (blocks["n_blocks"] == BOOTSTRAP_N_UNITS).any()
    ),
    f"headline = {BOOTSTRAP_N_UNITS} blocks",
)
# The point estimate is a property of the predictions, not of the blocking, so it must be
# identical across rows and equal to the headline matrix value.
headline_pr_auc = float(mat.loc[("baseline_tessera", "xgboost", 0), "pr_auc__pooled"])
worst, detail = max_abs_diff([(v, headline_pr_auc) for v in blocks["pr_auc"]])
check(
    "Block-size table: the pooled PR-AUC is invariant to blocking and matches the "
    "performance matrix",
    worst < 1e-9,
    detail,
)

block_rows = []
for _, r in blocks.sort_values("n_blocks").iterrows():
    n = int(r["n_blocks"])
    label = f"\\textbf{{{n}}}" if n == BOOTSTRAP_N_UNITS else str(n)
    source = BLOCK_SOURCE.get(r["block_source"], r["block_source"])
    if n == BOOTSTRAP_N_UNITS:
        source += " (headline)"
    block_rows.append(
        f"{label} & {source} & "
        + " & ".join(
            [
                f3(r["pr_auc"]),
                f3(r["boot_mean"]),
                f"[{f3(r['ci_lo'])}, {f3(r['ci_hi'])}]",
                f3(r["ci_width"]),
            ]
        )
        + r" \\"
    )

block_latex = (
    "\\begin{table}[H]\n\\centering\n"
    "\\caption{tab:sup\\_block\\_size\\_sensitivity}\n"
    "\\label{tab:sup_block_size_sensitivity}\n\\small\n\\setlength{\\tabcolsep}{6pt}\n"
    "\\resizebox{\\textwidth}{!}{%\n\\begin{threeparttable}\n"
    "\\begin{tabular}{rlcccc}\n\\toprule\n"
    "Blocks & Block source & Pooled PR-AUC & Bootstrap mean & 95\\% interval & "
    "Interval width \\\\\n\\midrule\n" + "\n".join(block_rows) + "\n\\bottomrule\n\\end{tabular}\n"
    "\\end{threeparttable}%\n}\n\\end{table}\n"
)
write_tex(
    "tab:sup_block_size_sensitivity",
    "Supplementary table - Bootstrap block-count sensitivity",
    block_latex,
    show=False,
)

In [ ]:
# Supplementary table: spread of inner-fold parcel PR-AUC across hyperparameter search
# trials. Alongside the overall spread, the top-5, top-10 and top-20 ranges describe how
# flat the search is near its optimum - the evidence that hyperparameter selection is
# unlikely to bias the reported comparisons.
GAP_COLUMNS = [
    ("selected_worst", "Selected $-$ worst", 3),
    ("selected_median", "Selected $-$ median", 3),
    ("top5_range", "Top-5 range", 3),
    ("top10_range", "Top-10 range", 3),
    ("top20_range", "Top-20 range", 3),
]


def hp_gaps(arch, fs):
    """Per-outer-fold search statistics for one configuration."""
    run = headline_run(arch, fs)
    rows = []
    for f in FOLD_IDS:
        trials = pd.read_csv(run / f"hp_trials_fold{f}.csv")
        scores = trials.loc[trials["state"] == "COMPLETE", "pr_auc"].astype(float)
        ordered = np.sort(scores.to_numpy())[::-1]
        selected = ordered[0]
        rows.append(
            {
                "n_trials": len(ordered),
                "selected_worst": selected - ordered[-1],
                "selected_median": selected - float(np.median(ordered)),
                "top5_range": selected - ordered[min(4, len(ordered) - 1)],
                "top10_range": selected - ordered[min(9, len(ordered) - 1)],
                "top20_range": selected - ordered[min(19, len(ordered) - 1)],
            }
        )
    return pd.DataFrame(rows)


def gap_summary(frames):
    """Median [minimum, maximum] of each statistic over the pooled searches."""
    pooled = pd.concat(frames, ignore_index=True)
    cells = []
    for column, _label, dp in GAP_COLUMNS:
        values = pooled[column]
        cells.append(f"{values.median():.{dp}f} [{values.min():.{dp}f}, {values.max():.{dp}f}]")
    return len(pooled), cells


s4_rows = []
family_sizes = {}
family_budget = {}
for family, archs in [("XGBoost", ["xgboost"]), ("CNN", ["cnn_3x3", "cnn_5x5", "cnn_7x7"])]:
    per_fs = {fs: [hp_gaps(a, fs) for a in archs] for fs in FS_ORDER}
    all_frames = [g for frames in per_fs.values() for g in frames]
    n_all, cells_all = gap_summary(all_frames)
    family_sizes[family] = (n_all, len(archs) * len(FS_ORDER) * len(FOLD_IDS))
    family_budget[family] = sorted(int(v) for v in pd.concat(all_frames)["n_trials"].unique())
    s4_rows.append(
        f"{family} & \\textit{{All feature sets}} & {n_all} & " + " & ".join(cells_all) + r" \\"
    )
    for fs in FS_ORDER:
        n_fs, cells_fs = gap_summary(per_fs[fs])
        s4_rows.append(f" & {FS_SHORT[fs]} & {n_fs} & " + " & ".join(cells_fs) + r" \\")
    if family == "XGBoost":
        s4_rows.append("\\midrule")

check(
    "HP-gap table: one search summarised per configuration x outer fold",
    all(n == expected for n, expected in family_sizes.values()),
    ", ".join(f"{fam}: {n}/{e}" for fam, (n, e) in family_sizes.items()),
)
check(
    "HP-gap table: the trial budget is constant within each family",
    all(len(b) == 1 for b in family_budget.values()),
    ", ".join(f"{fam}: {b}" for fam, b in family_budget.items()),
)

s4_latex = (
    "\\begin{table}[H]\n\\centering\n"
    "\\caption{tab:sup\\_hp\\_gaps}\n"
    "\\label{tab:sup_hp_gaps}\n\\small\n\\setlength{\\tabcolsep}{4pt}\n"
    # Portrait: the tabular measures ~640 pt against a ~484 pt text block, so it is
    # scaled to \textwidth like the other wide supplementary tables.
    "\\resizebox{\\textwidth}{!}{%\n\\begin{threeparttable}\n"
    "\\begin{tabular}{llrccccc}\n\\toprule\n"
    "Family & Feature set & $n$ & "
    + " & ".join(label for _c, label, _d in GAP_COLUMNS)
    + " \\\\\n\\midrule\n"
    + "\n".join(s4_rows)
    + "\n\\bottomrule\n\\end{tabular}\n"
    "\\end{threeparttable}%\n}\n\\end{table}\n"
)
write_tex(
    "tab:sup_hp_gaps", "Supplementary table - Hyperparameter-search spread", s4_latex, show=False
)

In [ ]:
# Supplementary table: the numbers behind the sampling-mode sensitivity figure
# (pr_auc_by_mode.pdf), extended with ROC-AUC and F1 at both the parcel and the pixel
# support. The sweep refits each fold with the headline run's hyperparameters and no
# inner Optuna search, so no nested inner threshold exists for it: its "inner" operating
# point is the 0.5 fallback of utils.cv.FoldResult.from_predictions, labelled as such.
SAMPLING_RUN = sorted(p for p in (paths.results / "sampling_sensitivity").iterdir() if p.is_dir())[
    -1
]
sampling = pd.read_csv(SAMPLING_RUN / "sampling_sensitivity.csv")
SAMPLING_MODES = [
    ("none", "None (all sampled pixels)"),
    ("inverse_prevalence", "Inverse prevalence"),
    ("stratified_forest_type", "Stratified by forest type"),
]
LEVELS = [("parcel", "Parcel"), ("pixel", "Pixel")]
sampling_idx = sampling.set_index(["sampling_mode", "level", "metric"])


def sampling_value(mode, level, metric, column):
    return float(sampling_idx.loc[(mode, level, metric), column])


check(
    "Sampling table: every sampling mode is scored at both supports",
    all(
        (mode, level, metric) in sampling_idx.index
        for mode, _l in SAMPLING_MODES
        for level, _d in LEVELS
        for metric in ("pr_auc", "roc_auc", "f1", "f1_inner_threshold")
    ),
    f"{len(SAMPLING_MODES)} modes x {len(LEVELS)} levels, run {SAMPLING_RUN.name}",
)

# The 'none' mode reuses the headline hyperparameters and sampling, so it must reproduce
# the headline nested-CV pooled metrics exactly - an independent cross-source check.
headline_parcel = mat.loc[("baseline_tessera", "xgboost", 0)]
headline_pixel = pd.read_csv(
    headline_run("xgboost", "baseline_tessera") / "pooled_pixel_metrics.csv"
).set_index("metric")["value"]
worst, detail = max_abs_diff(
    [
        (sampling_value("none", "parcel", "pr_auc", "pooled"), headline_parcel["pr_auc__pooled"]),
        (sampling_value("none", "parcel", "roc_auc", "pooled"), headline_parcel["roc_auc__pooled"]),
        (sampling_value("none", "pixel", "pr_auc", "pooled"), headline_pixel["pr_auc"]),
        (sampling_value("none", "pixel", "roc_auc", "pooled"), headline_pixel["roc_auc"]),
    ]
)
check(
    "Sampling table: the 'none' mode reproduces the headline nested-CV pooled metrics",
    worst < 1e-9,
    detail,
)

# Every mode's inner operating point is the 0.5 fallback (no inner search in this sweep).
fallbacks = {
    sampling_value(mode, level, "inner_threshold", "mean")
    for mode, _l in SAMPLING_MODES
    for level, _d in LEVELS
}
check(
    "Sampling table: the sweep's inner operating point is the fixed 0.5 fallback",
    fallbacks == {0.5},
    f"thresholds observed: {sorted(fallbacks)}",
)

sampling_rows = []
for mode, mode_label in SAMPLING_MODES:
    for i, (level, level_label) in enumerate(LEVELS):
        cells = [
            f3(sampling_value(mode, level, "pr_auc", "pooled")),
            f3(sampling_value(mode, level, "roc_auc", "pooled")),
            f3(sampling_value(mode, level, "f1", "pooled")),
        ]
        for metric in ("pr_auc", "roc_auc"):
            mean = sampling_value(mode, level, metric, "mean")
            lo = sampling_value(mode, level, metric, "min")
            hi = sampling_value(mode, level, metric, "max")
            cells.append(f"{f3(mean)} [{f3(lo)}, {f3(hi)}]")
        label = mode_label if i == 0 else ""
        sampling_rows.append(f"{label} & {level_label} & " + " & ".join(cells) + r" \\")
    if mode != SAMPLING_MODES[-1][0]:
        sampling_rows.append("\\midrule")

sampling_latex = (
    "\\begin{table}[H]\n\\centering\n"
    "\\caption{tab:sup\\_sampling\\_sensitivity}\n"
    "\\label{tab:sup_sampling_sensitivity}\n\\small\n\\setlength{\\tabcolsep}{4pt}\n"
    "\\resizebox{\\textwidth}{!}{%\n\\begin{threeparttable}\n"
    "\\begin{tabular}{llccccc}\n\\toprule\n"
    " & & \\multicolumn{3}{c}{Pooled out-of-fold} & \\multicolumn{2}{c}{Across folds: "
    "mean [min, max]} \\\\\n\\cmidrule(lr){3-5}\\cmidrule(lr){6-7}\n"
    "Sampling mode & Level & PR-AUC & ROC-AUC & $F_1$ & PR-AUC & ROC-AUC "
    "\\\\\n\\midrule\n" + "\n".join(sampling_rows) + "\n\\bottomrule\n\\end{tabular}\n"
    "\\end{threeparttable}%\n}\n\\end{table}\n"
)
write_tex(
    "tab:sup_sampling_sensitivity",
    "Supplementary table - Sampling-mode sensitivity",
    sampling_latex,
    show=False,
)

In [ ]:
# Supplementary table: pooled out-of-fold pixel-level performance at 0 km, reporting the
# operating point at BOTH the pooled outer F1-maximising threshold and the per-fold
# inner-fold thresholds. Validation: the threshold-free metrics and the outer-threshold
# block are recomputed from the pixel parquets and compared to pooled_pixel_metrics.csv.
def pixel_stats(arch, fs):
    run = headline_run(arch, fs)
    pooled_csv = pd.read_csv(run / "pooled_pixel_metrics.csv").set_index("metric")["value"]
    pfm = pd.read_csv(run / "per_fold_metrics.csv")
    pfm = pfm[pfm["level"] == "pixel"].set_index("outer_fold")
    parts, binary_inner = [], []
    for f in FOLD_IDS:
        frame = pd.read_parquet(run / f"predictions_fold{f}.parquet", columns=["y_true", "p"])
        parts.append(frame)
        binary_inner.append(frame["p"].to_numpy() >= float(pfm.loc[f, "inner_threshold"]))
    pooled = pd.concat(parts, ignore_index=True)
    y = pooled["y_true"].to_numpy()
    p = pooled["p"].to_numpy()
    inner_pred = np.concatenate(binary_inner)
    outer_pred = p >= float(pooled_csv["threshold"])
    weights = pfm["n"].astype(float)
    return {
        "n": int(pooled_csv["n"]),
        "prevalence": float(pooled_csv["prevalence"]),
        "pr_auc": float(pooled_csv["pr_auc"]),
        "roc_auc": float(pooled_csv["roc_auc"]),
        "outer_thr": float(pooled_csv["threshold"]),
        "outer_f1": float(pooled_csv["f1"]),
        "outer_precision": float(pooled_csv["precision"]),
        "outer_recall": float(pooled_csv["recall"]),
        "inner_thr": float((pfm["inner_threshold"] * weights).sum() / weights.sum()),
        "inner_f1": float(f1_score(y, inner_pred)),
        "inner_precision": float(precision_score(y, inner_pred)),
        "inner_recall": float(recall_score(y, inner_pred)),
        "_recomputed": (
            float(average_precision_score(y, p)),
            float(roc_auc_score(y, p)),
            float(f1_score(y, outer_pred)),
        ),
    }


print("[pixel table] pooling pixel predictions for 16 configurations (large parquets)...")
pixel_rows = {(fs, a): pixel_stats(a, fs) for fs in FS_ORDER for a in ARCH_ORDER}

pairs = []
for s in pixel_rows.values():
    pr, roc, f1_outer = s["_recomputed"]
    pairs += [(pr, s["pr_auc"]), (roc, s["roc_auc"]), (f1_outer, s["outer_f1"])]
worst, detail = max_abs_diff(pairs)
check(
    "Pixel table: PR/ROC-AUC and outer F1 recomputed from parquets match "
    "pooled_pixel_metrics.csv",
    worst < 1e-9,
    detail,
)

s5_rows = []
for fs in FS_ORDER:
    for a in ARCH_ORDER:
        s = pixel_rows[(fs, a)]
        label = FS_SHORT[fs] if a == "xgboost" else ""
        s5_rows.append(
            f"{label} & {ARCH_TEX[a]} & "
            + " & ".join(
                [
                    f3(s["pr_auc"]),
                    f3(s["roc_auc"]),
                    f3(s["outer_f1"]),
                    f3(s["outer_precision"]),
                    f3(s["outer_recall"]),
                    f3(s["outer_thr"]),
                    f3(s["inner_f1"]),
                    f3(s["inner_precision"]),
                    f3(s["inner_recall"]),
                    f3(s["inner_thr"]),
                ]
            )
            + r" \\"
        )
    if fs != FS_ORDER[-1]:
        s5_rows.append("\\midrule")

n_min = min(s["n"] for s in pixel_rows.values())
n_max = max(s["n"] for s in pixel_rows.values())
prev = pixel_rows[("baseline", "xgboost")]["prevalence"]
s5_latex = (
    "\\begin{table}[htbp]\n\\centering\n"
    "\\caption{tab:sup\\_pixel\\_performance\\_matrix}\n"
    "\\label{tab:sup_pixel_performance_matrix}\n\\small\n"
    "\\setlength{\\tabcolsep}{5pt}\n\\renewcommand{\\arraystretch}{0.95}\n"
    "\\resizebox{\\textwidth}{!}{%\n\\begin{threeparttable}\n"
    "\\begin{tabular}{llcccccccccc}\n\\toprule\n"
    " & & \\multicolumn{2}{c}{Threshold-free} & \\multicolumn{4}{c}{Outer pooled "
    "threshold} & \\multicolumn{4}{c}{Inner-fold threshold} \\\\\n"
    "\\cmidrule(lr){3-4}\\cmidrule(lr){5-8}\\cmidrule(lr){9-12}\n"
    "Feature set & Architecture & PR-AUC & ROC-AUC & $F_1$ & Precision & Recall & Thresh. "
    "& $F_1$ & Precision & Recall & Thresh. \\\\\n\\midrule\n"
    + "\n".join(s5_rows)
    + "\n\\bottomrule\n\\end{tabular}\n"
    "\\end{threeparttable}%\n}\n\\end{table}\n"
)
write_tex(
    "tab:sup_pixel_performance_matrix",
    "Supplementary table - Pixel-level performance matrix",
    s5_latex,
    show=False,
)

s5_show = pd.DataFrame(
    [
        {
            "feature_set": fs,
            "architecture": a,
            **{k: v for k, v in pixel_rows[(fs, a)].items() if not k.startswith("_")},
        }
        for fs in FS_ORDER
        for a in ARCH_ORDER
    ]
)
print(s5_show.round(3).to_string(index=False))

In [ ]:
# Supplementary tables: per-fold out-of-fold parcel PR-AUC, ROC-AUC and $F_1$ for every
# configuration and buffer, with the coordinate-only control. $F_1$ uses each fold's own
# inner-fold decision threshold (the operating point reported everywhere else), so the
# per-fold values are directly comparable with the pooled estimates. Validation: every
# per-fold value is recomputed from the saved parcel predictions and compared with the
# performance-matrix CSV.
FOLD_METRICS = [
    ("pr_auc", "PR-AUC", "tab:sup_per_fold_pr_auc"),
    ("roc_auc", "ROC-AUC", "tab:sup_per_fold_roc_auc"),
    ("f1_inner", "$F_1$", "tab:sup_per_fold_f1"),
]


def fold_values(arch, fs, buffer_km):
    """{metric: {fold: value}} for one configuration, from the saved predictions."""
    frames = parcel_frames(arch, fs, buffer_km)
    thresholds = inner_thresholds(arch, fs)
    out = {metric: {} for metric, _label, _tab in FOLD_METRICS}
    for f, frame in frames.items():
        y = frame["y_true"].to_numpy()
        p = frame["p_mean"].to_numpy()
        out["pr_auc"][f] = float(average_precision_score(y, p))
        out["roc_auc"][f] = float(roc_auc_score(y, p))
        out["f1_inner"][f] = float(f1_score(y, p >= thresholds[f]))
    return out


print("[per-fold tables] recomputing per-fold metrics from parcel predictions...")
pairs = []
recomputed = {}
for fs in FS_ORDER:
    for a in ARCH_ORDER:
        for buffer_km in BUFFERS:
            values = fold_values(a, fs, buffer_km)
            row = mat.loc[(fs, a, buffer_km)]
            for metric, _label, _tab in FOLD_METRICS:
                recomputed[(metric, fs, a, buffer_km)] = values[metric]
                pairs += [(values[metric][f], row[f"{metric}__fold{f}"]) for f in FOLD_IDS]
worst, detail = max_abs_diff(pairs)
check(
    "Per-fold tables: every fold PR-AUC, ROC-AUC and $F_1$ recomputed from predictions "
    "matches the matrix",
    worst < 1e-9,
    detail,
)

xy_folds = {}
for buffer_km in BUFFERS:
    xy_folds[("xgboost", buffer_km)] = fold_values("xgboost", XY, buffer_km)
for arch in ARCH_ORDER[1:]:
    xy_folds[(arch, 0)] = fold_values(arch, XY, 0)


def fold_cells(folds):
    arr = np.array([folds[f] for f in FOLD_IDS])
    return [f3(v) for v in [*arr, arr.mean(), arr.std(ddof=1), arr.max() - arr.min()]]


def per_fold_latex(metric, label):
    rows = []
    for fs in FS_ORDER:
        for a in ARCH_ORDER:
            for buffer_km in BUFFERS:
                fs_label = FS_SHORT[fs] if (a == "xgboost" and buffer_km == 0) else ""
                arch_label = ARCH_TEX[a] if buffer_km == 0 else ""
                cells = fold_cells(recomputed[(metric, fs, a, buffer_km)])
                rows.append(
                    f"{fs_label} & {arch_label} & {buffer_km} & " + " & ".join(cells) + r" \\"
                )
        rows.append("\\midrule")
    for buffer_km in BUFFERS:
        fs_label = FS_SHORT[XY] if buffer_km == 0 else ""
        arch_label = ARCH_TEX["xgboost"] if buffer_km == 0 else ""
        cells = fold_cells(xy_folds[("xgboost", buffer_km)][metric])
        rows.append(f"{fs_label} & {arch_label} & {buffer_km} & " + " & ".join(cells) + r" \\")
    for arch in ARCH_ORDER[1:]:
        cells = fold_cells(xy_folds[(arch, 0)][metric])
        rows.append(f" & {ARCH_TEX[arch]} & 0 & " + " & ".join(cells) + r" \\")
    return (
        "\\begin{table}[H]\n\\centering\n" + tex_caption(label) + "\n"
        f"\\label{{{label}}}\n\\footnotesize\n\\setlength{{\\tabcolsep}}{{4pt}}\n"
        "\\resizebox{\\textwidth}{!}{%\n\\begin{threeparttable}\n"
        "\\begin{tabular}{llrrrrrrrrrr}\n\\toprule\n"
        " & & & \\multicolumn{6}{c}{Outer fold} & & & \\\\\n\\cmidrule(lr){4-9}\n"
        "Feature set & Architecture & Buffer & 1 & 2 & 3 & 4 & 5 & 6 & Mean & SD & "
        "Range \\\\\n& & (km) & & & & & & & & & \\\\\n\\midrule\n"
        + "\n".join(rows)
        + "\n\\bottomrule\n\\end{tabular}\n"
        "\\end{threeparttable}%\n}\n\\end{table}\n"
    )


for metric, _metric_label, label in FOLD_METRICS:
    plain = {"pr_auc": "PR-AUC", "roc_auc": "ROC-AUC", "f1_inner": "F1"}[metric]
    write_tex(
        label,
        f"Supplementary table - Per-fold parcel {plain}",
        per_fold_latex(metric, label),
        show=False,
    )

In [ ]:
# Supplementary table (landscape): pooled out-of-fold parcel performance with 95%
# intervals for every configuration and buffer, at BOTH the inner-fold and the pooled
# outer decision threshold, plus the coordinate-only control. It runs to several pages,
# so it is a longtable with a continuation header; a longtable cannot be wrapped in
# \resizebox (a resize box cannot break across pages), so the horizontal fit is set by
# the font size and column separation and verified to leave no overfull box.
# Validation: pooled PR/ROC-AUC recomputed from the saved predictions against the matrix.
from utils.reporting import metric_matrix_row

S7_METRICS = [
    "pr_auc",
    "roc_auc",
    "f1_inner",
    "precision_inner",
    "recall_inner",
    "f1_outer",
    "precision_outer",
    "recall_outer",
]

pairs = []
for fs in FS_ORDER:
    for a in ARCH_ORDER:
        for buffer_km in BUFFERS:
            pooled = pooled_frame(parcel_frames(a, fs, buffer_km))
            y, p = pooled["y_true"].to_numpy(), pooled["p_mean"].to_numpy()
            row = mat.loc[(fs, a, buffer_km)]
            pairs.append((average_precision_score(y, p), row["pr_auc__pooled"]))
            pairs.append((roc_auc_score(y, p), row["roc_auc__pooled"]))
worst, detail = max_abs_diff(pairs)
check(
    "Pooled matrix: PR/ROC-AUC recomputed from predictions match the matrix CSV",
    worst < 1e-9,
    detail,
)

print(
    "[pooled matrix] bootstrapping the coordinate-only XGBoost rows " "(3 buffers x 5000 reps)..."
)
xy_inner = inner_thresholds("xgboost", XY)
xy_rows = {}
for buffer_km in BUFFERS:
    xy_rows[("xgboost", buffer_km)] = metric_matrix_row(
        parcel_frames("xgboost", XY, buffer_km),
        xy_inner,
        BLOCK_ASSIGNMENT,
        reps=BOOTSTRAP_REPS_CROSS_CONFIG,
        seed=SEED,
        n_jobs=8,
    )
cnn_xy = pd.concat(
    [pd.read_csv(f) for f in sorted((paths.results / "coordinate_cnn_buffer").glob("cnn_*.csv"))],
    ignore_index=True,
)
for _, r in cnn_xy.iterrows():
    xy_rows[(r["architecture"], int(r["buffer_km"]))] = r


def s7_cells(row):
    return [ci3(row[f"{m}__pooled"], row[f"{m}__ci_lo"], row[f"{m}__ci_hi"]) for m in S7_METRICS]


s7_rows = []
for fs in [*FS_ORDER, XY]:
    for a in ARCH_ORDER:
        for buffer_km in BUFFERS:
            fs_label = FS_SHORT[fs] if (a == "xgboost" and buffer_km == 0) else ""
            arch_label = ARCH_TEX[a] if buffer_km == 0 else ""
            if fs != XY:
                cells = s7_cells(mat.loc[(fs, a, buffer_km)])
            elif (a, buffer_km) in xy_rows:
                cells = s7_cells(xy_rows[(a, buffer_km)])
            else:
                cells = ["--"] * len(S7_METRICS)
            s7_rows.append(
                f"{fs_label} & {arch_label} & {buffer_km} & "
                + " & ".join(texnum(c) if c != "--" else c for c in cells)
                + r" \\"
            )
    if fs != XY:
        s7_rows.append("\\midrule")

S7_HEADER = (
    " & & & \\multicolumn{2}{c}{Threshold-free} & \\multicolumn{3}{c}{Inner-fold "
    "threshold} & \\multicolumn{3}{c}{Outer pooled threshold} \\\\\n"
    "\\cmidrule(lr){4-5}\\cmidrule(lr){6-8}\\cmidrule(lr){9-11}\n"
    # Every heading sits on one line: at \scriptsize with 2 pt column separation the
    # tabular measures ~708 pt against the ~731 pt landscape text block.
    "Feature set & Architecture & Buffer (km) & PR-AUC & ROC-AUC & $F_1$ & Precision & "
    "Recall & $F_1$ & Precision & Recall \\\\\n\\midrule\n"
)
s7_latex = (
    # \scriptsize at 2pt column separation fits the landscape text width with room
    # to spare and runs to two pages; a longtable cannot be \resizebox-ed.
    "\\begin{landscape}\n\\begingroup\n\\scriptsize\n"
    "\\setlength{\\tabcolsep}{2pt}\n\\renewcommand{\\arraystretch}{0.92}\n"
    "\\setlength{\\aboverulesep}{1pt}\n\\setlength{\\belowrulesep}{1pt}\n"
    "\\begin{longtable}{llrcccccccc}\n"
    "\\caption{tab:sup\\_performance\\_matrix}\\label{tab:sup_performance_matrix}\\\\\n\\toprule\n"
    + S7_HEADER
    + "\\endfirsthead\n"
    "\\multicolumn{11}{l}{\\footnotesize\\textit{Table "
    "\\ref{tab:sup_performance_matrix} continued from the previous page.}}\\\\\n"
    "\\toprule\n" + S7_HEADER + "\\endhead\n\\midrule\n"
    "\\multicolumn{11}{r}{\\footnotesize\\textit{Continued on the next page.}}\\\\\n"
    "\\endfoot\n\\bottomrule\n\\endlastfoot\n" + "\n".join(s7_rows) + "\n\\end{longtable}\n"
    "\\endgroup\n\\end{landscape}\n"
)
write_tex(
    "tab:sup_performance_matrix",
    "Supplementary table - Pooled parcel performance matrix",
    s7_latex,
    show=False,
)

In [ ]:
# Supplementary tables: paired contrasts. The feature-set family gains a
# 'Baseline - Coordinate-only' row per buffer (same paired block bootstrap), so each
# Validation: contrast point estimates against differences of the matrix rows.
from utils.reporting import paired_row

C_METRICS = ["pr_auc", "roc_auc", "f1_inner", "precision_inner", "recall_inner"]
fs_ct = pd.read_csv(PM_RUN / "feature_set_contrasts.csv")
emb_ct = pd.read_csv(PM_RUN / "embedding_contrasts.csv")
arch_ct = pd.read_csv(PM_RUN / "architecture_contrasts.csv")

CONTRAST_TEX = {
    "baseline_conventional_eo - baseline": "+ Conventional EO $-$ Baseline",
    "baseline_alphaearth - baseline": "+ AlphaEarth $-$ Baseline",
    "baseline_tessera - baseline": "+ TESSERA $-$ Baseline",
    "baseline_alphaearth - baseline_conventional_eo": "+ AlphaEarth $-$ + Conventional EO",
    "baseline_tessera - baseline_conventional_eo": "+ TESSERA $-$ + Conventional EO",
    "baseline_tessera - baseline_alphaearth": "+ TESSERA $-$ + AlphaEarth",
    "baseline - xy_coords": "Baseline $-$ Coordinate-only",
}
FS_CONTRASTS = [
    "baseline_conventional_eo - baseline",
    "baseline_alphaearth - baseline",
    "baseline_tessera - baseline",
]
EMB_CONTRASTS = [
    "baseline_alphaearth - baseline_conventional_eo",
    "baseline_tessera - baseline_conventional_eo",
    "baseline_tessera - baseline_alphaearth",
]

# Check 1: each contrast's pooled PR-AUC equals the difference of the two matrix rows.
pairs = []
for table in (fs_ct, emb_ct):
    for _, r in table.iterrows():
        set_a, set_b = r["contrast"].split(" - ")
        a_val = mat.loc[(set_a, "xgboost", r["buffer_km"]), "pr_auc__pooled"]
        b_val = mat.loc[(set_b, "xgboost", r["buffer_km"]), "pr_auc__pooled"]
        pairs.append((r["pr_auc__pooled"], a_val - b_val))
for _, r in arch_ct.iterrows():
    left, right = r["contrast"].split(" - ")
    arch_a, fs_a = left.split("__")
    arch_b, fs_b = right.split("__")
    a_val = mat.loc[(fs_a, arch_a, r["buffer_km"]), "pr_auc__pooled"]
    b_val = mat.loc[(fs_b, arch_b, r["buffer_km"]), "pr_auc__pooled"]
    pairs.append((r["pr_auc__pooled"], a_val - b_val))
worst, detail = max_abs_diff(pairs)
check(
    "Contrast tables: pooled PR-AUC differences equal matrix-row differences", worst < 1e-9, detail
)


def ci3_aligned(v, lo, hi):
    """``ci3`` with a phantom minus in front of every non-negative number.

    The contrast columns are centred and their values are signed, so a cell whose
    numbers carry no minus is narrower than one that does and its digits sit off the
    line of the rows above. Padding each non-negative number with a phantom minus makes
    every cell the same width, so centring lines the digits up. The padding costs no
    column width, because each column is already as wide as its all-negative cells, and
    ``texnum`` converts the phantom "-" to "$-$" along with the real ones.
    """

    def pad(x):
        return "" if float(x) < 0 else "\\phantom{-}"

    return f"{pad(v)}{f3(v)} [{pad(lo)}{f3(lo)}, {pad(hi)}{f3(hi)}]"


def p_cell(p):
    """Two-sided bootstrap p value, bold when it is below 0.05 (as is the PR-AUC contrast)."""
    return f"\\textbf{{{pfmt(p)}}}" if p < 0.05 else pfmt(p)


def contrast_cells(row, bold_pr):
    cells = []
    for m in C_METRICS:
        cell = texnum(ci3_aligned(row[f"{m}__pooled"], row[f"{m}__ci_lo"], row[f"{m}__ci_hi"]))
        if m == "pr_auc" and bold_pr:
            cell = f"\\textbf{{{cell}}}"
        cells.append(cell)
    return cells


def contrast_table_rows(table, order):
    rows = []
    for i, contrast in enumerate(order):
        for buffer_km in BUFFERS:
            r = table[(table["contrast"] == contrast) & (table["buffer_km"] == buffer_km)]
            if r.empty:
                continue
            r = r.iloc[0]
            p = r["pr_auc__p"]
            cells = contrast_cells(r, p < 0.05)
            label = CONTRAST_TEX[contrast] if buffer_km == BUFFERS[0] else ""
            rows.append(
                f"{label} & {buffer_km} & " + " & ".join(cells) + f" & {p_cell(p)}" + r" \\"
            )
        if i < len(order) - 1:
            rows.append("\\midrule")
    return rows


# Extend the feature-set family with the Baseline - Coordinate-only contrast.
print("[contrasts] paired bootstrap for Baseline - Coordinate-only " "(3 buffers x 5000 reps)...")
base_inner = inner_thresholds("xgboost", "baseline")
xy_rows_ct = []
for buffer_km in BUFFERS:
    row = paired_row(
        parcel_frames("xgboost", "baseline", buffer_km),
        parcel_frames("xgboost", XY, buffer_km),
        base_inner,
        xy_inner,
        BLOCK_ASSIGNMENT,
        reps=BOOTSTRAP_REPS_CROSS_CONFIG,
        seed=SEED,
        n_jobs=8,
    )
    row.update({"contrast": "baseline - xy_coords", "buffer_km": buffer_km})
    xy_rows_ct.append(row)
fs_ct_ext = pd.concat([fs_ct, pd.DataFrame(xy_rows_ct)], ignore_index=True)
s8_rows = contrast_table_rows(fs_ct_ext, [*FS_CONTRASTS, "baseline - xy_coords"])


def contrast_latex(label, rows):
    return (
        "\\begin{table}[H]\n\\centering\n"
        + tex_caption(label)
        + f"\n\\label{{{label}}}\n\\scriptsize\n"
        "\\setlength{\\tabcolsep}{6pt}\n\\renewcommand{\\arraystretch}{0.92}\n"
        "\\resizebox{\\textwidth}{!}{%\n\\begin{threeparttable}\n"
        "\\begin{tabular}{lrcccccc}\n\\toprule\n"
        " &  & \\multicolumn{2}{c}{Threshold-free} & \\multicolumn{3}{c}{Inner-fold "
        "threshold} & \\\\\n\\cmidrule(lr){3-4}\\cmidrule(lr){5-7}\n"
        "Contrast & Buffer & PR-AUC & ROC-AUC & $F_1$ & Precision & Recall & "
        "$p$ \\\\\n & (km) &  &  &  &  &  & \\\\\n\\midrule\n"
        + "\n".join(rows)
        + "\n\\bottomrule\n\\end{tabular}\n"
        "\\end{threeparttable}%\n}\n\\end{table}\n"
    )


s8_latex = contrast_latex(
    "tab:sup_feature_set_contrasts",
    s8_rows,
)
write_tex(
    "tab:sup_feature_set_contrasts",
    "Supplementary table - Feature-set contrasts vs baseline and coordinate control",
    s8_latex,
    show=False,
)

s9_latex = contrast_latex(
    "tab:sup_embedding_contrasts",
    contrast_table_rows(emb_ct, EMB_CONTRASTS),
)
write_tex(
    "tab:sup_embedding_contrasts",
    "Supplementary table - EO-representation contrasts",
    s9_latex,
    show=False,
)

s10_rows = []
for i, fs in enumerate(FS_ORDER):
    for j, arch in enumerate(ARCH_ORDER[1:]):
        contrast = f"{arch}__{fs} - xgboost__{fs}"
        for buffer_km in BUFFERS:
            r = arch_ct[
                (arch_ct["contrast"] == contrast) & (arch_ct["buffer_km"] == buffer_km)
            ].iloc[0]
            cells = contrast_cells(r, r["pr_auc__p"] < 0.05)
            fs_label = FS_SHORT[fs] if (j == 0 and buffer_km == 0) else ""
            arch_label = ARCH_TEX[arch] if buffer_km == 0 else ""
            s10_rows.append(
                f"{fs_label} & {arch_label} & {buffer_km} & "
                + " & ".join(cells)
                + f" & {p_cell(r['pr_auc__p'])}"
                + r" \\"
            )
    if i < len(FS_ORDER) - 1:
        s10_rows.append("\\midrule")
s10_latex = (
    # Landscape rather than \resizebox: at \scriptsize the tabular measures ~640 pt
    # against a ~484 pt portrait text block, so scaling it to fit rendered the body at
    # about 5 pt. The landscape text block is ~731 pt and takes it unscaled.
    "\\begin{landscape}\n\\begin{table}[H]\n\\centering\n"
    "\\caption{tab:sup\\_architecture\\_contrasts}\n"
    "\\label{tab:sup_architecture_contrasts}\n\\scriptsize\n"
    "\\setlength{\\tabcolsep}{5pt}\n\\renewcommand{\\arraystretch}{0.92}\n"
    "\\begin{threeparttable}\n"
    "\\begin{tabular}{llrcccccc}\n\\toprule\n"
    " &  &  & \\multicolumn{2}{c}{Threshold-free} & \\multicolumn{3}{c}{Inner-fold "
    "threshold} & \\\\\n\\cmidrule(lr){4-5}\\cmidrule(lr){6-8}\n"
    "Feature set & Architecture & Buffer & PR-AUC & ROC-AUC & $F_1$ & Precision & Recall "
    "& $p$ \\\\\n &  & (km) &  &  &  &  &  & \\\\\n\\midrule\n"
    + "\n".join(s10_rows)
    + "\n\\bottomrule\n\\end{tabular}\n"
    "\\end{threeparttable}\n\\end{table}\n\\end{landscape}\n"
)
write_tex(
    "tab:sup_architecture_contrasts",
    "Supplementary table - Architecture contrasts (CNN vs XGBoost)",
    s10_latex,
    show=False,
)

In [ ]:
# Supplementary table: ROC-AUC of every product within strata of forest type, altitude
# and slope (same 20-block bootstrap as notebook 011). Validation: stratum coverage.
from utils.bootstrap import block_bootstrap_distribution, percentile_interval

ORDER = ["sabatini", "munteanu", "kathmann", "schickhofer", "ratsakatika"]
ev = pd.read_parquet(COMP_RUN / "product_eval_parcels.parquet")
ev = ev.merge(
    _labels_gdf[["parcel_id", "bootstrap_id", "corine_forest_type"]].assign(
        parcel_id=_labels_gdf["parcel_id"].astype("int64")
    ),
    on="parcel_id",
    how="left",
)
_idx = pd.read_parquet(NESTED / "pixel_index.parquet")
_feats = np.load(paths.cache / "pixel_features" / "baseline_tessera.npy", mmap_mode="r")
_bands = list(FEATURE_SET_BANDS["baseline_tessera"])
_cov = pd.DataFrame({"parcel_id": _idx["parcel_id"].to_numpy()})
for _name in ("elevation_m", "slope_deg"):
    _values = np.asarray(_feats[:, _bands.index(_name)], dtype=np.float64)
    _values[_values == NODATA] = np.nan
    _cov[_name] = _values
ev = ev.merge(_cov.groupby("parcel_id").mean(), on="parcel_id", how="left")


def _tertiles(series):
    q = series.quantile([0, 1 / 3, 2 / 3, 1]).to_numpy()
    labels = [f"{q[i]:.0f}-{q[i + 1]:.0f}" for i in range(3)]
    return (
        pd.cut(
            series, bins=[-np.inf, q[1], q[2], np.inf], labels=labels, include_lowest=True
        ).astype("object"),
        labels,
    )


ev["forest_type"] = ev["corine_forest_type"]
strata = [("Forest type", "forest_type", ["broadleaf", "coniferous", "mixed"])]
for name, title in [
    ("elevation_m", "Altitude tertile (m)"),
    ("slope_deg", "Slope tertile ($^\\circ$)"),
]:
    col = f"{name}_tertile"
    ev[col], labs = _tertiles(ev[name])
    strata.append((title, col, labs))

coverage = {title: int(ev[col].isin(values).sum()) for title, col, values in strata}
check(
    "Stratified table: tertile panels cover every evaluation parcel and the forest-type "
    "panel covers every CORINE-classified parcel",
    coverage["Altitude tertile (m)"] == len(ev)
    and coverage["Slope tertile ($^\\circ$)"] == len(ev)
    and coverage["Forest type"] == int(ev["forest_type"].notna().sum()),
    ", ".join(f"{t}: {n}/{len(ev)}" for t, n in coverage.items()),
)

SCORE_COL = {"ratsakatika": "ratsakatika_oof", **{s: f"ogf_{s}_frac" for s in COMPARISON_STUDIES}}
_ref = ev["reference_label"].astype(int).to_numpy()
print(
    "[stratified table] block bootstrap (9 strata x 5 products x "
    f"{BOOTSTRAP_REPS_INFERENCE} reps)..."
)
s11_rows = []
for title, col, values in strata:
    s11_rows.append(f"\\multicolumn{{8}}{{l}}{{\\textit{{{title}}}}}\\\\")
    for v in values:
        m = (ev[col] == v).to_numpy()
        y = _ref[m]
        scores = np.column_stack([ev.loc[m, SCORE_COL[k]].fillna(0.0).to_numpy() for k in ORDER])
        dist = block_bootstrap_distribution(
            y,
            scores,
            ev.loc[m, "bootstrap_id"].to_numpy(),
            lambda a, b: roc_auc_score(a, b),
            n_reps=BOOTSTRAP_REPS_INFERENCE,
            seed=SEED,
        )
        cells = []
        for j in range(len(ORDER)):
            lo, hi = percentile_interval(dist[:, j])
            cells.append(ci3(roc_auc_score(y, scores[:, j]), lo, hi))
        n_txt = f"{int(m.sum()):,}".replace(",", "{,}")
        display_v = v.replace("-", "--") if col != "forest_type" else v
        s11_rows.append(
            f"\\quad {display_v} & {n_txt} & {y.mean():.3f} & " + " & ".join(cells) + r" \\"
        )
    if title != strata[-1][0]:
        s11_rows.append("\\midrule")

s11_latex = (
    "\\begin{table}[H]\n\\centering\n"
    "\\caption{tab:sup\\_stratified\\_roc\\_auc}\n\\label{tab:sup_stratified_roc_auc}\n\\scriptsize\n"
    "\\setlength{\\tabcolsep}{2pt}\n"
    "\\resizebox{\\textwidth}{!}{%\n\\begin{threeparttable}\n"
    "\\begin{tabular}{lrrccccc}\n\\toprule\n"
    " & & & \\multicolumn{5}{c}{ROC-AUC [95\\% interval]} \\\\\n\\cmidrule(lr){4-8}\n"
    "Stratum & $n$ & Prev. & Sabatini & Munteanu et al. & Kathmann et al. & "
    "Schickhofer & \\textbf{This study} \\\\\n"
    " & & & (2020) & (2022) & (2017) & \\& Schwarz (2019) & \\\\\n\\midrule\n"
    + "\n".join(s11_rows)
    + "\n\\bottomrule\n\\end{tabular}\n"
    "\\end{threeparttable}%\n}\n\\end{table}\n"
)
write_tex(
    "tab:sup_stratified_roc_auc",
    "Supplementary table - Stratified ROC-AUC by forest type, altitude and slope",
    s11_latex,
    show=False,
)

In [ ]:
# Supplementary table: mapped old-growth area at parcel resolution vs each product's
# native resolution. Parcel areas come from the comparison gpkg verdicts; native areas
# from the products' own rasters/vectors clipped to the AOI
# (scripts.build_existing_product_masks; Dropbox raster reads are retried once).
import sys

if str(paths.repo_root) not in sys.path:
    sys.path.insert(0, str(paths.repo_root))
from scripts.build_existing_product_masks import native_ogf_area_ha

DISPLAY_S12 = {
    "sabatini": "Sabatini (2020)",
    "munteanu": "Munteanu et al. (2022)",
    "kathmann": "Kathmann et al. (2017)",
    "schickhofer": "Schickhofer and Schwarz (2019)",
}
aoi_gdf = gpd.read_file(paths.aoi)
raw_root = paths.repo_root / "data" / "raw" / "existing_products"
native_area = {}
for s in STUDY_ORDER:
    try:
        native_area[s] = float(native_ogf_area_ha(s, aoi_gdf, raw_root))
    except Exception:
        try:
            native_area[s] = float(native_ogf_area_ha(s, aoi_gdf, raw_root))
        except Exception as error:
            print(f"[error] native area for {s} could not be recomputed: {error}")
            native_area[s] = float("nan")
check(
    "Area table: native areas recomputed for every product",
    all(np.isfinite(v) for v in native_area.values()),
    ", ".join(
        f"{s}: {v:,.0f} ha" if np.isfinite(v) else f"{s}: failed" for s, v in native_area.items()
    ),
)

s12_rows = []
for s in STUDY_ORDER:
    p_ha, n_ha = parcel_area[s], native_area[s]
    diff = n_ha - p_ha
    cells = [
        f"{p_ha:,.0f}",
        f"{100 * p_ha / aoi_ha:.1f}",
        f"{n_ha:,.0f}" if np.isfinite(n_ha) else "--",
        f"{100 * n_ha / aoi_ha:.1f}" if np.isfinite(n_ha) else "--",
        f"{diff:+,.0f}" if np.isfinite(diff) else "--",
        f"{100 * diff / p_ha:+.1f}" if np.isfinite(diff) else "--",
    ]
    s12_rows.append(f"{DISPLAY_S12[s]} & " + " & ".join(cells) + r" \\")

s12_latex = (
    "\\begin{table}[H]\n\\centering\n"
    "\\caption{tab:sup\\_area\\_parcel\\_vs\\_native}\n\\label{tab:sup_area_parcel_vs_native}\n\\small\n"
    # Seven narrow columns fit the text block at \small, so this table is not resizeboxed.
    "\\begin{threeparttable}\n"
    "\\begin{tabular}{lrrrrrr}\n\\toprule\n"
    " & \\multicolumn{2}{c}{Parcel} & \\multicolumn{2}{c}{Native pixel-level} & "
    "\\multicolumn{2}{c}{Difference} \\\\\n"
    "\\cmidrule(lr){2-3}\\cmidrule(lr){4-5}\\cmidrule(lr){6-7}\n"
    "Product & (ha) & (\\% AOI) & (ha) & (\\% AOI) & (ha) & (\\%) \\\\\n\\midrule\n"
    + "\n".join(s12_rows)
    + "\n\\bottomrule\n\\end{tabular}\n"
    "\\end{threeparttable}\n\\end{table}\n"
)
write_tex(
    "tab:sup_area_parcel_vs_native",
    "Supplementary table - Parcel vs native mapped areas",
    s12_latex,
    show=False,
)

In [ ]:
# New supplementary tables evidencing the robustness checks of notebook
# 014_robustness_checks: leave-one-fold-out sensitivity, the seed sweep, and the
# unlabelled-stratum (excluded middle) analysis. These have no manuscript counterpart
# yet; internal consistency is validated against the performance matrix.
RB = paths.figures / "014_robustness_checks" / "tables"
lofo_pr = pd.read_csv(RB / "table1_lofo_config_pr_auc.csv", index_col=0)
lofo_ct = pd.read_csv(RB / "table3_lofo_contrast_pr_auc.csv", index_col=0)
seed_cfg = pd.read_csv(RB / "table5_seed_sweep_configs.csv", index_col=0)
seed_ct = pd.read_csv(RB / "table6_seed_sweep_contrasts.csv", index_col=0)
stratum = pd.read_csv(RB / "table7_area_by_stratum.csv", index_col=0)
agree = pd.read_csv(RB / "table8_unlabelled_agreement.csv", index_col=0)

# Internal check: the LOFO all-fold pooled PR-AUC must reproduce the performance matrix.
mat_idx = mat
label_of = {f"{FEATURE_SETS[fs]} ({b} km)": (fs, b) for fs in FS_ORDER for b in (0, 10)}
pairs = [
    (lofo_pr.loc[label, "all folds"], mat_idx.loc[(fs, "xgboost", b), "pr_auc__pooled"])
    for label, (fs, b) in label_of.items()
]
worst, detail = max_abs_diff(pairs)
check(
    "Robustness tables: LOFO all-fold pooled PR-AUC matches the performance matrix",
    worst < 5e-4,
    detail,
)


def escape(text):
    """Escape the LaTeX specials that can appear in a pandas index or column name."""
    out = str(text)
    for char in ("&", "%", "_", "#"):
        out = out.replace(char, "\\" + char)
    return out


def prepare(df, keep, rename=None):
    """Select and order the reported columns, then apply display names."""
    return df[list(keep)].rename(columns=dict(rename or {}))


def head_cell(text):
    """A column heading, stacked over two lines when it contains a newline."""
    parts = str(text).split("\n")
    if len(parts) == 1:
        return escape(text)
    return "\\shortstack{" + "\\\\".join(escape(part) for part in parts) + "}"


def df_table(df, label, fmts, colspec=None, index_head="", resize=True, head=None):
    cols = list(df.columns)
    colspec = colspec or ("l" + "c" * len(cols))
    header = (
        index_head + " & " + " & ".join(head_cell((head or {}).get(c, c)) for c in cols) + r" \\"
    )
    rows = []
    for idx, row in df.iterrows():
        cells = [fmts.get(c, str)(row[c]) for c in cols]
        rows.append(escape(idx) + " & " + " & ".join(texnum(str(c)) for c in cells) + r" \\")
    return (
        "\\begin{table}[H]\n\\centering\n"
        + tex_caption(label)
        + f"\n\\label{{{label}}}\n\\footnotesize\n"
        "\\setlength{\\tabcolsep}{4pt}\n"
        + ("\\resizebox{\\textwidth}{!}{%\n" if resize else "")
        + "\\begin{threeparttable}\n"
        f"\\begin{{tabular}}{{{colspec}}}\n\\toprule\n"
        + header
        + "\n\\midrule\n"
        + "\n".join(rows)
        + "\n\\bottomrule\n\\end{tabular}\n"
        + ("\\end{threeparttable}%\n}\n" if resize else "\\end{threeparttable}\n")
        + "\\end{table}\n"
    )


def fnum(dp):
    return lambda v: ("--" if pd.isna(pd.to_numeric(v, errors="coerce")) else f"{float(v):.{dp}f}")


# Reported columns: the jackknife SE is dropped (with n = 6 the range is the honest
# statistic), and the headings are capitalised for the manuscript.
LOFO_KEEP = ["all folds", *[f"excl. fold {k}" for k in FOLD_IDS], "range"]
LOFO_RENAME = {
    "all folds": "All folds",
    "range": "Range",
    **{f"excl. fold {k}": f"Excl. fold {k}" for k in FOLD_IDS},
}
lofo_pr = prepare(lofo_pr, LOFO_KEEP, LOFO_RENAME)


def bracket_contrast(label):
    """Bracket each side of an ``A - B (n km)`` contrast label.

    Feature-set names contain "+" and spaces, so "Baseline + AlphaEarth - Baseline + EO"
    gives no visual cue where the first set ends and the second begins. Bracketing each
    side makes the subtraction unambiguous.

    The label is wrapped in braces because it becomes the first cell of a row: a row that
    begins with "[" directly after the previous row's "\\\\" is read as the optional
    line-break argument "\\\\[<dimen>]", which fails with "Missing number" and "Illegal
    unit of measure". The braces make the row start with "{" instead.
    """
    head, sep, tail = str(label).rpartition(" (")
    head = head if sep else str(label)
    left, minus, right = head.partition(" - ")
    if not minus:
        return str(label)
    bracketed = f"{{[{left}] $-$ [{right}]}}"
    return f"{bracketed} ({tail}" if sep else bracketed


lofo_ct = prepare(lofo_ct, LOFO_KEEP, LOFO_RENAME)
lofo_ct.index = [bracket_contrast(i) for i in lofo_ct.index]

lofo_fmt = dict.fromkeys(lofo_pr.columns, fnum(3))
latex = df_table(
    lofo_pr,
    "tab:sup_lofo_pr_auc",
    lofo_fmt,
    index_head="Configuration",
)
write_tex(
    "tab:sup_lofo_pr_auc", "Supplementary table - Leave-one-fold-out PR-AUC", latex, show=False
)

latex = df_table(
    lofo_ct,
    "tab:sup_lofo_contrasts",
    dict.fromkeys(lofo_ct.columns, fnum(3)),
    index_head="Contrast",
)
write_tex(
    "tab:sup_lofo_contrasts",
    "Supplementary table - Leave-one-fold-out contrasts",
    latex,
    show=False,
)

# Reported columns: the paper seed, then the sweep size and the between-seed summary.
# The evaluation bootstrap interval is reported in the performance matrix instead.
SEED_KEEP = ["paper seed", "n_seeds", "seed mean", "between-seed SD", "seed min-max"]
SEED_RENAME = {
    "paper seed": "Paper seed",
    "n_seeds": "$n$ seeds",
    "seed mean": "Seed mean",
    "between-seed SD": "Seed SD",
    "seed min-max": "Seed min max",
}
seed_cfg = prepare(seed_cfg, SEED_KEEP, SEED_RENAME)
seed_ct = prepare(seed_ct, SEED_KEEP, SEED_RENAME)
seed_ct.index = [bracket_contrast(i) for i in seed_ct.index]
seed_fmt = {
    "Paper seed": fnum(3),
    "Seed mean": fnum(3),
    "Seed SD": fnum(3),
    "Seed min max": str,
    "$n$ seeds": lambda v: str(int(v)),
}
latex = df_table(
    seed_cfg,
    "tab:sup_seed_sweep",
    seed_fmt,
    index_head="Configuration",
    resize=False,
)
write_tex(
    "tab:sup_seed_sweep", "Supplementary table - Seed-sweep configurations", latex, show=False
)

latex = df_table(
    seed_ct,
    "tab:sup_seed_sweep_contrasts",
    seed_fmt,
    index_head="Contrast",
    resize=False,
)
write_tex(
    "tab:sup_seed_sweep_contrasts", "Supplementary table - Seed-sweep contrasts", latex, show=False
)

# Reported columns: the calibrated-probability column is dropped and the headings
# capitalised for the manuscript.
STRATUM_KEEP = [
    "parcels",
    "area (ha)",
    "OGF-classified parcels",
    "OGF-classified share (%)",
    "OGF-classified area (ha)",
    "share of mapped OGF area (%)",
]
stratum = prepare(
    stratum,
    STRATUM_KEEP,
    {
        "parcels": "Parcels",
        "area (ha)": "Area (ha)",
        "share of mapped OGF area (%)": "Share of mapped OGF area (%)",
    },
)
stratum_fmt = {
    "Parcels": lambda v: f"{int(v):,}",
    "Area (ha)": lambda v: f"{float(v):,.0f}",
    "OGF-classified parcels": lambda v: f"{int(v):,}",
    "OGF-classified share (%)": fnum(1),
    "OGF-classified area (ha)": lambda v: f"{float(v):,.0f}",
    "Share of mapped OGF area (%)": fnum(1),
}
# Notebook 014 measures parcel area as the mapped 10 m pixels the parcel contains -- the
# convention of the headline mapped area -- so this table's OGF total is the 53,896 ha of
# main Table 3 exactly, not a polygon-area figure a fraction of a percent above it. Checked
# below rather than assumed.
check(
    "Area-by-stratum table: the OGF total matches the headline mapped area",
    abs(float(stratum.loc["All parcels", "OGF-classified area (ha)"]) - study_area_ha) < 0.5,
    f"{stratum.loc['All parcels', 'OGF-classified area (ha)']:,.1f} vs {study_area_ha:,.1f} ha",
)
latex = df_table(
    stratum,
    "tab:sup_area_by_stratum",
    stratum_fmt,
    index_head="Stratum",
    resize=False,
    head={
        "OGF-classified parcels": "OGF-classified\nparcels",
        "OGF-classified share (%)": "OGF-classified\nshare (%)",
        "OGF-classified area (ha)": "OGF-classified\narea (ha)",
        "Share of mapped OGF area (%)": "Share of mapped\nOGF area (%)",
    },
)
write_tex(
    "tab:sup_area_by_stratum", "Supplementary table - Mapped OGF area by stratum", latex, show=False
)

# Reported columns: the both-flagged share, kappa and the calibrated probability are
# dropped; "ours" becomes "This study" to match the rest of the manuscript.
AGREE_KEEP = [
    "parcels",
    "area (ha)",
    "ours OGF (%)",
    "PRIMOFARO OGF (%)",
    "overall agreement (%)",
    "positive agreement (%)",
    "negative agreement (%)",
]
agree = prepare(
    agree,
    AGREE_KEEP,
    {
        "parcels": "Parcels",
        "area (ha)": "Area (ha)",
        "ours OGF (%)": "This study OGF (%)",
        "PRIMOFARO OGF (%)": "Schickhofer and Schwarz OGF (%)",
        "overall agreement (%)": "Overall agreement (%)",
        # Both statistics are set overlaps (intersection over union) on one class, so the
        # headings say "overlap" rather than the conventional positive/negative agreement.
        "positive agreement (%)": "OGF overlap (%)",
        "negative agreement (%)": "Non-OGF overlap (%)",
    },
)
agree_fmt = {
    "Parcels": lambda v: f"{int(v):,}",
    "Area (ha)": lambda v: f"{float(v):,.0f}",
    "This study OGF (%)": fnum(1),
    "Schickhofer and Schwarz OGF (%)": fnum(1),
    "Overall agreement (%)": fnum(1),
    "OGF overlap (%)": fnum(1),
    "Non-OGF overlap (%)": fnum(1),
}
latex = df_table(
    agree,
    "tab:sup_unlabelled_agreement",
    agree_fmt,
    index_head="Stratum",
    # Eight columns overrun the supplementary text block by ~14 pt at full size, so this
    # table scales to \textwidth like the rest (its earlier nine-column form did too).
    head={
        "This study OGF (%)": "This study\nOGF (%)",
        "Schickhofer and Schwarz OGF (%)": "Schickhofer and\nSchwarz OGF (%)",
        "Overall agreement (%)": "Overall\nagreement (%)",
        "OGF overlap (%)": "OGF\noverlap (%)",
        "Non-OGF overlap (%)": "Non-OGF\noverlap (%)",
    },
)
write_tex(
    "tab:sup_unlabelled_agreement",
    "Supplementary table - Unlabelled-parcel agreement",
    latex,
    show=False,
)
print(
    "[robustness] six new supplementary tables emitted "
    "(no manuscript counterparts to validate against)."
)

In [ ]:
# Consolidated validation report: every internal-consistency check run above.
report = pd.DataFrame(CHECKS)
print(f"LaTeX snippets for every table: {LATEX_DIR}")
print(f"\n{int(report['passed'].sum())}/{len(report)} checks passed\n")
print(report.to_string(index=False))
if not report["passed"].all():
    print("\nFailed checks:")
    for _, r in report[~report["passed"]].iterrows():
        print(f"  - {r['check']} ({r['detail']})")